In [1]:
import json
import csv
import re
import statistics

In [ ]:
import json

INPUT_FILE = "directer_predictions.json"
OUTPUT_FILE = "directer_predictions_100_5.json"

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

first_50 = data[:50]

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(first_50, f, indent=4, ensure_ascii=False)

print(f"Saved {len(first_50)} entries to {OUTPUT_FILE}")

Saved 50 entries to directer_predictions_100_5.json


In [2]:
# ============================================================
# EXACT METRIC FROM evaluations.ipynb — DO NOT CHANGE
# ============================================================

def _tokenize(text):
    return set(re.findall(r"[a-zA-Z0-9]+", text.lower()))


def category_prf(pred_text, gold_phrases):
    """Token-overlap precision/recall/F1."""

    gold_words = set()

    for phrase in gold_phrases:
        gold_words |= _tokenize(phrase)

    pred_words = _tokenize(pred_text)

    if not gold_words and not pred_words:
        return 1.0, 1.0, 1.0

    if not pred_words:
        return 0.0, 0.0, 0.0

    if not gold_words:
        return 0.0, 1.0, 0.0

    inter = len(gold_words & pred_words)

    precision = inter / len(pred_words)
    recall = inter / len(gold_words)

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    return precision, recall, f1


# ============================================================
# INPUT / OUTPUT FILES
# ============================================================

INPUT_FILE = "directer_predictions_100_5_1-5.json"

PER_ENTRY_OUTPUT = "director_evaluation_100_5_1-5_per_entry.csv"
SUMMARY_OUTPUT = "director_evaluation_100_5_1-5_summary.json"


# ============================================================
# PICO MAPPING
#
# Ground truth key  -> Prediction key
# ============================================================

PICO_MAPPING = {
    "Population": {
        "letter": "P",
        "gold_key": "population",
        "prediction_key": "Population",
    },

    "Intervention": {
        "letter": "I",
        "gold_key": "intervention_exposure",
        "prediction_key": "Intervention",
    },

    "Comparator": {
        "letter": "C",
        "gold_key": "comparison",
        "prediction_key": "Comparator",
    },

    "Outcome": {
        "letter": "O",
        "gold_key": "outcome",
        "prediction_key": "Outcome",
    },
}


# ============================================================
# EVALUATE ONE RECORD
# ============================================================

def evaluate_record(record, index):

    predicted_pico = record.get("pico_directer", {})

    result = {
        "index": index,
        "title": record.get("title", "")
    }

    for category, config in PICO_MAPPING.items():

        letter = config["letter"]
        gold_key = config["gold_key"]
        prediction_key = config["prediction_key"]

        # ----------------------------------------------------
        # GROUND TRUTH
        # ----------------------------------------------------

        gold_phrases = record.get(gold_key, [])

        if gold_phrases is None:
            gold_phrases = []

        if isinstance(gold_phrases, str):
            gold_phrases = [gold_phrases]

        # ----------------------------------------------------
        # PREDICTION
        # ----------------------------------------------------

        predicted_phrases = predicted_pico.get(
            prediction_key,
            []
        )

        if predicted_phrases is None:
            predicted_phrases = []

        if isinstance(predicted_phrases, str):
            predicted_phrases = [predicted_phrases]

        # Convert prediction list into one text string
        predicted_text = " ".join(predicted_phrases)

        # ----------------------------------------------------
        # EXACT METRIC
        # ----------------------------------------------------

        precision, recall, f1 = category_prf(
            predicted_text,
            gold_phrases
        )

        # ----------------------------------------------------
        # SAVE GOLD / PREDICTION
        # ----------------------------------------------------

        result[f"{letter}_gold"] = " | ".join(gold_phrases)

        result[f"{letter}_pred"] = " | ".join(predicted_phrases)

        # ----------------------------------------------------
        # SAVE METRICS
        # ----------------------------------------------------

        result[f"{letter}_precision"] = precision
        result[f"{letter}_recall"] = recall
        result[f"{letter}_f1"] = f1

    return result


# ============================================================
# LOAD JSON
# ============================================================

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)


print(f"Loaded records: {len(data)}")


# ============================================================
# EVALUATE ALL RECORDS
# ============================================================

rows = []

for index, record in enumerate(data, start=1):

    try:
        row = evaluate_record(record, index)
        rows.append(row)

    except Exception as e:
        print(f"Error processing record {index}: {e}")


print(f"Successfully evaluated: {len(rows)}")


# ============================================================
# SAVE PER-ENTRY RESULTS
# ============================================================

fieldnames = [
    "index",
    "title",

    # Population
    "P_gold",
    "P_pred",
    "P_precision",
    "P_recall",
    "P_f1",

    # Intervention
    "I_gold",
    "I_pred",
    "I_precision",
    "I_recall",
    "I_f1",

    # Comparator
    "C_gold",
    "C_pred",
    "C_precision",
    "C_recall",
    "C_f1",

    # Outcome
    "O_gold",
    "O_pred",
    "O_precision",
    "O_recall",
    "O_f1",
]


with open(
    PER_ENTRY_OUTPUT,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=fieldnames
    )

    writer.writeheader()
    writer.writerows(rows)


print(f"\nPer-entry results saved to:")
print(PER_ENTRY_OUTPUT)


# ============================================================
# CALCULATE AVERAGES
# ============================================================

categories = ["P", "I", "C", "O"]

summary = {
    "num_records": len(rows),
    "categories": {},
}


# ------------------------------------------------------------
# Average separately for P / I / C / O
# ------------------------------------------------------------

for letter in categories:

    precision_values = [
        row[f"{letter}_precision"]
        for row in rows
    ]

    recall_values = [
        row[f"{letter}_recall"]
        for row in rows
    ]

    f1_values = [
        row[f"{letter}_f1"]
        for row in rows
    ]

    summary["categories"][letter] = {
        "precision": statistics.mean(precision_values),
        "recall": statistics.mean(recall_values),
        "f1": statistics.mean(f1_values),
    }


# ============================================================
# OVERALL AVERAGE
#
# Macro-average across all 4 PICO categories.
# Each record contributes P, I, C and O equally.
# ============================================================

all_precision = []
all_recall = []
all_f1 = []

for row in rows:

    for letter in categories:

        all_precision.append(
            row[f"{letter}_precision"]
        )

        all_recall.append(
            row[f"{letter}_recall"]
        )

        all_f1.append(
            row[f"{letter}_f1"]
        )


summary["overall"] = {
    "precision": statistics.mean(all_precision),
    "recall": statistics.mean(all_recall),
    "f1": statistics.mean(all_f1),
}


# ============================================================
# SAVE SUMMARY
# ============================================================

with open(
    SUMMARY_OUTPUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )


# ============================================================
# PRINT RESULTS
# ============================================================

print("\n" + "=" * 60)
print("DIRECTOR EVALUATION RESULTS")
print("=" * 60)

for letter, category in [
    ("P", "Population"),
    ("I", "Intervention"),
    ("C", "Comparator"),
    ("O", "Outcome"),
]:

    metrics = summary["categories"][letter]

    print(f"\n{category}")
    print(f"  Precision : {metrics['precision']:.4f}")
    print(f"  Recall    : {metrics['recall']:.4f}")
    print(f"  F1        : {metrics['f1']:.4f}")


print("\n" + "-" * 60)
print("OVERALL MACRO AVERAGE")
print("-" * 60)

print(
    f"Precision : "
    f"{summary['overall']['precision']:.4f}"
)

print(
    f"Recall    : "
    f"{summary['overall']['recall']:.4f}"
)

print(
    f"F1        : "
    f"{summary['overall']['f1']:.4f}"
)

print("\n" + "=" * 60)

print(f"\nSummary saved to:")
print(SUMMARY_OUTPUT)

Loaded records: 50
Successfully evaluated: 50

Per-entry results saved to:
director_evaluation_100_5_1-5_per_entry.csv

DIRECTOR EVALUATION RESULTS

Population
  Precision : 0.4944
  Recall    : 0.7368
  F1        : 0.4530

Intervention
  Precision : 0.5267
  Recall    : 0.6350
  F1        : 0.5040

Comparator
  Precision : 0.2472
  Recall    : 0.9000
  F1        : 0.2525

Outcome
  Precision : 0.4752
  Recall    : 0.8156
  F1        : 0.4709

------------------------------------------------------------
OVERALL MACRO AVERAGE
------------------------------------------------------------
Precision : 0.4358
Recall    : 0.7719
F1        : 0.4201


Summary saved to:
director_evaluation_100_5_1-5_summary.json
